In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.609500                    0.554700   
             precision                   0.523344                    0.458796   
             recall                      0.647510                    0.589883   
             f1                          0.577940                    0.515020   
             kappa                       0.222661                    0.115859   
             MCC                         0.227474                    0.119356   
outputsTest  accuracy                    0.601800                    0.550300   
             precision                   0.513738                    0.450393   
             recall                      0.636159                    0.586717   
             f1                          0.567744                    0.508609   
             kappa                       0.206440                    0.107262   
             MCC                         0.210898                    0.110678   
outputsAll   accuracy                    0.619800                    0.554100   
             precision                   0.533977                    0.457137   
             recall                      0.655808                    0.588310   
             f1                          0.587656                    0.513080   
             kappa                       0.242149                    0.114133   
             MCC                         0.247142                    0.117924   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.614400   
             precision                         0.664052   
             recall                            0.675786   
             f1                                0.669135   
             kappa                             0.206103   
             MCC                               0.206811   
outputsTest  accuracy                          0.617300   
             precision                         0.667949   
             recall                            0.674740   
             f1                                0.670377   
             kappa                             0.213545   
             MCC                               0.214491   
outputsAll   accuracy                          0.617800   
             precision                         0.668115   
             recall                            0.673216   
             f1                                0.669887   
             kappa                             0.215379   
             MCC                               0.216061   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.502800   
             precision                      0.356234   
             recall                         0.513687   
             f1                             0.419691   
             kappa                          0.009318   
             MCC                            0.010238   
outputsTest  accuracy                       0.502300   
             precision                      0.359810   
             recall                         0.510475   
             f1                             0.421093   
             kappa                          0.007716   
             MCC                            0.008013   
outputsAll   accuracy                       0.503200   
             precision                      0.354042   
             recall                         0.514207   
             f1                             0.418529   
             kappa                          0.010225   
             MCC                            0.010652   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.620300                  0.426000  
             precision                  0.313199                  0.116573  
             recall                     0.637899                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.228505,0.115986,0.212455,0.009634,0.199342,-0.184262
accuracy,0.610367,0.553033,0.616500,0.502767,0.613833,0.425200
f1,0.577780,0.512236,0.669799,0.419771,0.412761,0.167478
kappa,0.223750,0.112418,0.211676,0.009086,0.174158,-0.143235
precision,0.523686,0.455442,0.666705,0.356695,0.308682,0.115166
recall,0.646492,0.588303,0.674581,0.512790,0.631588,0.312561


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.289282                    0.052919   
             spearman                   0.286887                    0.086741   
             MSE                        1.421437                    1.894162   
             RMSE                       1.189963                    1.374678   
             MAE                        0.956577                    1.050072   
outputsTest  pearson                    0.272533                    0.038471   
             spearman                   0.264642                    0.074444   
             MSE                        1.454934                    1.923058   
             RMSE                       1.204526                    1.385228   
             MAE                        0.965671                    1.059135   
outputsAll   pearson                    0.304286                    0.044255   
             spearman                   0.299952                    0.086021   
             MSE                        1.391428                    1.911490   
             RMSE                       1.177244                    1.381171   
             MAE                        0.940810                    1.053040   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.158849   
             spearman                         0.248914   
             MSE                              1.682302   
             RMSE                             1.294963   
             MAE                              0.983067   
outputsTest  pearson                          0.150480   
             spearman                         0.251954   
             MSE                              1.699040   
             RMSE                             1.301226   
             MAE                              0.985184   
outputsAll   pearson                          0.155843   
             spearman                         0.255832   
             MSE                              1.688314   
             RMSE                             1.296836   
             MAE                              0.981149   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.019763                  0.137122   
             spearman                      0.123518                  0.219796   
             MSE                           1.960474                  1.725755   
             RMSE                          1.398482                  1.311420   
             MAE                           1.046010                  0.963882   
outputsTest  pearson                       0.015260                  0.124795   
             spearman                      0.121180                  0.211197   
             MSE                           1.969480                  1.750410   
             RMSE                          1.401434                  1.320568   
             MAE                           1.056626                  0.974368   
outputsAll   pearson                       0.026876                  0.123494   
             spearman                      0.128809                  0.210510   
             MSE                           1.946248                  1.753013   
             RMSE                          1.393380                  1.321563   
             MAE                           1.047062                  0.970056   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.088019  
             spearman                 -0.184263  
             MSE                       2.176038  
             RMSE                      1.473797  
             MAE                       1.160179  
outputsTest  pearson                  -0.070620  
             spearman                 -0.175633  
             MSE                       2.141240  
             RMSE                      1.461900  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.954353,1.054082,0.983133,1.049900,0.969436,1.164458
MSE,1.422600,1.909570,1.689885,1.958734,1.743059,2.163359
RMSE,1.190578,1.380359,1.297675,1.397765,1.317850,1.469482
pearson,0.288700,0.045215,0.155057,0.020633,0.128470,-0.081680
spearman,0.283827,0.082402,0.252233,0.124502,0.213834,-0.176058
